Copyright (C) 2026 Michael Nowotny

This program is free software; you can redistribute it and/or modify
it under the terms of the GNU General Public License version 2 as
published by the Free Software Foundation.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
GNU General Public License for more details.

# What's New in PyJAGS v2.3.0

PyJAGS v2.3.0 is an API modernization release, bringing ideas from the
best Bayesian packages in the Python ecosystem (PyMC, CmdStanPy, emcee,
ArviZ) to JAGS users.

**New features covered in this notebook:**

1. `check_model()` -- syntax validation without compilation
2. `seed=` -- reproducible sampling via a single integer
3. `Model.__repr__` -- informative model display
4. Sampler diagnostics -- `samplers`, `is_adapted`, `iteration`
5. `iter_sample()` -- generator-based sampling with live convergence monitoring
6. `sample_more()` -- extend existing samples
7. Richer `from_pyjags()` -- observed data, constant data, metadata
8. `pathlib.Path` support
9. Convergence warnings
10. PEP 561 `py.typed` marker

## Setup

In [ ]:
import pyjags
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
from pathlib import Path
import tempfile

We will use a simple normal model throughout, so the focus stays on the
new API features rather than model complexity.

In [ ]:
MODEL_CODE = """
model {
    mu ~ dnorm(0, 0.001)
    sigma ~ dunif(0, 100)
    tau <- pow(sigma, -2)
    for (i in 1:N) {
        y[i] ~ dnorm(mu, tau)
    }
}
"""

np.random.seed(123)
y = np.random.normal(loc=10.0, scale=3.0, size=50)
data = dict(y=y, N=len(y))

---
## 1. Syntax Validation with `check_model()`

Validate JAGS model syntax without compiling or providing data. This is
unique to JAGS -- no other Bayesian package offers standalone syntax
checking.

In [ ]:
# Valid model
pyjags.check_model(code=MODEL_CODE)

In [ ]:
# Invalid model -- shows annotated error with source context
try:
    pyjags.check_model(code="model { x ~ not_a_distribution() }")
except Exception as e:
    print(f"Syntax error: {e}")

In [ ]:
# Also works with pathlib.Path
model_file = Path(tempfile.mktemp(suffix=".bug"))
model_file.write_text(MODEL_CODE)

pyjags.check_model(file=model_file)
model_file.unlink()

---
## 2. Reproducible Sampling with `seed=`

Pass a single integer seed to `Model()` for fully reproducible results.
Internally, PyJAGS uses `numpy.random.SeedSequence` to derive
statistically independent seeds for each chain and cycles through
JAGS's four built-in RNG factories for additional structural
independence.

Inspired by CmdStanPy's `seed=` and PyMC's `random_seed=`.

In [ ]:
m1 = pyjags.Model(
    code=MODEL_CODE, data=data, chains=4,
    adapt=1000, progress_bar=False, seed=42,
)
m1.sample(1000, vars=[])
s1 = m1.sample(1000, vars=["mu"])

m2 = pyjags.Model(
    code=MODEL_CODE, data=data, chains=4,
    adapt=1000, progress_bar=False, seed=42,
)
m2.sample(1000, vars=[])
s2 = m2.sample(1000, vars=["mu"])

print(f"Same seed, same results: {np.array_equal(s1['mu'], s2['mu'])}")

The `seed` parameter can be combined with non-RNG `init` values (e.g.,
starting values for parameters), but will raise a `ValueError` if the
`init` dictionary contains `.RNG.name` or `.RNG.seed` keys to prevent
conflicts.

---
## 3. Model Inspection: `__repr__` and Diagnostics

Models now have an informative `repr` and expose sampler diagnostics as
properties.

In [ ]:
model = pyjags.Model(
    code=MODEL_CODE, data=data, chains=4,
    adapt=1000, progress_bar=False,
)

# Rich repr
model

In [ ]:
# Sampler diagnostics
print(f"Adapted:   {model.is_adapted}")
print(f"Iteration: {model.iteration}")
print(f"\nSamplers used by JAGS:")
for s in model.samplers:
    print(f"  {s}")

---
## 4. Generator-Based Sampling with `iter_sample()`

Inspired by [emcee](https://emcee.readthedocs.io/)'s `sample()`
generator. Instead of blocking until all samples are drawn,
`iter_sample()` yields `SamplingState` objects at regular intervals,
letting you:

- Monitor convergence in real time
- Break early when convergence is achieved
- Build progress indicators

The `SamplingState` carries **accumulated** samples (not just the latest
chunk) and computes ESS and R-hat **lazily** -- diagnostics are only
calculated when you access them.

In [ ]:
model = pyjags.Model(
    code=MODEL_CODE, data=data, chains=4,
    adapt=1000, progress_bar=False,
)
model.sample(1000, vars=[])  # burn-in

# Sample in chunks of 500, monitoring convergence
for state in model.iter_sample(iterations=5000, chunk_size=500, vars=["mu", "sigma"]):
    mu_rhat = state.rhat["mu"]
    mu_ess = state.ess["mu"]
    print(
        f"Iteration {state.iteration:5d} | "
        f"mu R-hat: {mu_rhat:.4f} | "
        f"mu ESS: {mu_ess:7.0f}"
    )

In [ ]:
# Samples accumulate across chunks
print(f"Final samples shape: {state.samples['mu'].shape}")

### Early Stopping

Set a generous upper bound on iterations and break early when
convergence criteria are met:

In [ ]:
model = pyjags.Model(
    code=MODEL_CODE, data=data, chains=4,
    adapt=1000, progress_bar=False,
)
model.sample(1000, vars=[])  # burn-in

for state in model.iter_sample(iterations=50000, chunk_size=500, vars=["mu", "sigma"]):
    all_rhat_ok = all(v < 1.01 for v in state.rhat.values())
    all_ess_ok = all(v > 400 for v in state.ess.values())

    if all_rhat_ok and all_ess_ok:
        print(f"Converged at iteration {state.iteration}!")
        print(f"  R-hat: { {k: round(v, 4) for k, v in state.rhat.items()} }")
        print(f"  ESS:   { {k: round(v) for k, v in state.ess.items()} }")
        break

# Use state.samples for downstream analysis
idata = pyjags.from_pyjags(state.samples)
az.summary(idata)

---
## 5. Extending Samples with `sample_more()`

JAGS retains chain state between `sample()` calls, so drawing more
samples is trivial. `sample_more()` draws additional iterations and
concatenates them with your existing samples.

In [ ]:
model = pyjags.Model(
    code=MODEL_CODE, data=data, chains=4,
    adapt=1000, progress_bar=False,
)
model.sample(1000, vars=[])  # burn-in

# Start with 1000 samples
samples = model.sample(1000, vars=["mu", "sigma"])
print(f"Initial:       {samples['mu'].shape}")

# Need more? Extend:
samples = model.sample_more(2000, samples, vars=["mu", "sigma"])
print(f"After +2000:   {samples['mu'].shape}")

# Extend again:
samples = model.sample_more(1000, samples, vars=["mu", "sigma"])
print(f"After +1000:   {samples['mu'].shape}")

---
## 6. Richer ArviZ Integration

`from_pyjags()` now supports `observed_data` and `constant_data`
groups, and automatically sets metadata attributes following ArviZ
conventions.

In [ ]:
model = pyjags.Model(
    code=MODEL_CODE, data=data, chains=4,
    adapt=1000, progress_bar=False,
)
model.sample(1000, vars=[])
samples = model.sample(5000, vars=["mu", "sigma"])

idata = pyjags.from_pyjags(
    samples,
    observed_data={"y": y},
    constant_data={"N": np.array(len(y))},
)

# Metadata
print(f"Inference library: {idata.attrs['inference_library']}")
print(f"Version:           {idata.attrs.get('inference_library_version', 'unknown')}")
print(f"DataTree groups:   {[str(n) for n in idata.children]}")

In [ ]:
# Convenience wrapper: summary directly from samples
pyjags.summary(samples, var_names=["mu", "sigma"])

---
## 7. `pathlib.Path` Support

Both `Model()` and `check_model()` now accept `pathlib.Path` objects
in addition to string paths.

In [ ]:
model_path = Path(tempfile.mktemp(suffix=".bug"))
model_path.write_text(MODEL_CODE)

# Use pathlib.Path directly
model = pyjags.Model(
    file=model_path,
    data=data,
    chains=4,
    adapt=1000,
    progress_bar=False,
)
print(model)

# check_model also accepts pathlib
print(f"Syntax valid: {pyjags.check_model(file=model_path)}")

model_path.unlink()

---
## 8. Convergence Warnings

The `sample()` method now accepts a `warn_convergence` parameter.
When enabled, it checks R-hat and ESS after sampling and emits a
warning if convergence looks problematic. This is off by default to
avoid noise during burn-in.

In [ ]:
model = pyjags.Model(
    code=MODEL_CODE, data=data, chains=4,
    adapt=1000, progress_bar=False,
)
model.sample(1000, vars=[])  # burn-in

# Opt-in convergence warnings
samples = model.sample(5000, vars=["mu", "sigma"], warn_convergence=True)
print("Sampling complete (no warnings means convergence looks good)")

---
## 9. PEP 561 Type Checking Support

PyJAGS now ships a `py.typed` marker file (PEP 561), which tells type
checkers like mypy, pyright, and IDEs like PyCharm and VS Code that
PyJAGS supports inline type annotations.

This means you get autocompletion and type checking out of the box
when using PyJAGS in a type-aware editor. PyJAGS is ahead of PyMC in
this regard -- PyMC does not currently ship a `py.typed` marker.

In [ ]:
import importlib.resources

py_typed = importlib.resources.files("pyjags") / "py.typed"
print(f"py.typed exists: {py_typed.is_file()}")

---
## Summary

| Feature | API | Inspired by |
|---------|-----|-------------|
| Syntax validation | `pyjags.check_model()` | Unique to JAGS |
| Reproducible seeds | `Model(seed=42)` | CmdStanPy, PyMC |
| Model inspection | `repr(model)`, `.samplers`, `.is_adapted`, `.iteration` | PyMC, CmdStanPy |
| Generator sampling | `model.iter_sample()` | emcee |
| Extend samples | `model.sample_more()` | JAGS chain state retention |
| Richer ArviZ | `from_pyjags(observed_data=..., constant_data=...)` | ArviZ conventions |
| pathlib support | `Model(file=Path(...))` | Python standard library |
| Convergence warnings | `model.sample(warn_convergence=True)` | PyMC |
| Type checking | `py.typed` marker | CmdStanPy |